In [1]:
import fitz

doc = fitz.open("invoice.pdf")
page = doc[0]

# Zoom x3 for better quality (higher resolution)
zoom = 3
mat = fitz.Matrix(zoom, zoom)
pix = page.get_pixmap(matrix=mat)
pix.save("page1_hd.png")

print(f"High-res image saved! Size: {pix.width}x{pix.height}")


High-res image saved! Size: 1735x2474


In [2]:
import base64

with open("page1_hd.png", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

print("Image converted!")

Image converted!


In [3]:
from mistralai.client import Mistral

client = Mistral(api_key="ta_clé_ici")

response = client.chat.complete(
    model="pixtral-12b-2409",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Look carefully at this invoice image. Extract ONLY the data that is clearly visible. Do NOT invent or repeat lines. Respond ONLY in JSON format like this: {\"invoice_number\": \"\", \"date\": \"\", \"supplier\": \"\", \"client\": \"\", \"total_ht\": 0.0, \"total_tva\": 0.0, \"total_ttc\": 0.0}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

UnicodeEncodeError: 'ascii' codec can't encode character '\xe9' in position 12: ordinal not in range(128)

In [4]:
from mistralai.client import Mistral

client = Mistral(api_key="s46f3EZ1Up9LrY0INTDQ8wyGMSYggC05")

response = client.chat.complete(
    model="pixtral-12b-2409",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Look carefully at this invoice image. Extract ONLY the data that is clearly visible. Do NOT invent or repeat lines. Respond ONLY in JSON format like this: {\"invoice_number\": \"\", \"date\": \"\", \"supplier\": \"\", \"client\": \"\", \"total_ht\": 0.0, \"total_tva\": 0.0, \"total_ttc\": 0.0}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

```json
{
  "invoice_number": "1253888565",
  "date": "27.04.2026",
  "supplier": {
    "name": "SYSCO France SAS",
    "address": "ZONE IDF COMMERCIALE CS 30041 - 76201 Dieppe Cedex",
    "phone": "01.69.11.67.24",
    "fax": "01.69.11.40.95",
    "iban": "FR76 3000 401328 0010 9434 4804",
    "siret": "35422013000019",
    "capital": "100.000,00 Euro"
  },
  "client": {
    "reference": "354220",
    "facture": "130",
    "type_produit": "FACTURE"
  },
  "total_ht": 526.47,
  "total_tva": 28.96,
  "total_ttc": 555.43
}
```


In [5]:
import json

raw = response.choices[0].message.content
clean = raw.replace("```json", "").replace("```", "").strip()

data = json.loads(clean)
print(data)
print(f"\nTotal to pay: {data['total_ttc']}€")

{'invoice_number': '1253888565', 'date': '27.04.2026', 'supplier': {'name': 'SYSCO France SAS', 'address': 'ZONE IDF COMMERCIALE CS 30041 - 76201 Dieppe Cedex', 'phone': '01.69.11.67.24', 'fax': '01.69.11.40.95', 'iban': 'FR76 3000 401328 0010 9434 4804', 'siret': '35422013000019', 'capital': '100.000,00 Euro'}, 'client': {'reference': '354220', 'facture': '130', 'type_produit': 'FACTURE'}, 'total_ht': 526.47, 'total_tva': 28.96, 'total_ttc': 555.43}

Total to pay: 555.43€
